In [1]:
import sys
sys.path.insert(0, '../lib')

In [2]:
import os
import pathlib
import typing
import datetime
import json

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import decoupler
import statsmodels.stats.multitest
import scanpy as sc
import upsetplot
import scipy.stats

import common_plots
import common_data
import pb_utils

/projects/b1196/envs/serniczek/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
common_plots.setup_plotting()

In [4]:
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200
%config InlineBackend.figure_format = "retina"

In [5]:
def sanitize_name(name):
    return name.replace(' ', '_').replace('*', '').replace(';', '_and').replace('/', '_')

# HALLMARK gene sets enrichment in pseudobulks

We want a heatmap: 
- rows are HALLMARK processes
- columns are samples
- values are GSVA enrichment

In [6]:
msigdb = decoupler.get_resource('MSigDB', organism='human')

In [7]:
hallmark = msigdb[msigdb.collection.eq('hallmark')]

In [8]:
hallmark = hallmark[~hallmark.duplicated(['geneset', 'genesymbol'])]

In [9]:
ROOT = common_data.DATA

In [ ]:
BASE = ROOT / '05_pseudobulk/01_all_pseudobulk'

In [ ]:
OUTDIR = common_data.ROOT / '08_website/explore_gsva'

In [12]:
os.makedirs(OUTDIR, exist_ok=True)

In [13]:
sc_labels = pd.read_csv(common_data.SC_LABELS, index_col=0)
cat_covs = common_data.get_sc_categorical_covariates()

In [14]:
sc_labels = sc_labels.merge(cat_covs.loc[:, ['Sex']], left_on='bal_barcode', right_index=True)

In [15]:
ehr_data = pd.read_csv(common_data.CLINICAL, index_col=0)

In [16]:
sc_labels = sc_labels.merge(
    ehr_data,
    left_index=True,
    right_index=True,
    how='left',
    suffixes=('', '_ehr')
)

Copy from 00_LABELS, we didn't export raw pathogen names there

In [17]:
PATHOGEN_NAMES = {
    'Viridans Streptococcus': 'Viridans streptococcus',
    'enterobacter cloacae complex': 'Enterobacter cloacae',
    'escherichia coli': 'Escherichia coli',
    'haemophilus influenzae': 'Haemophilus influenzae',
    'klebsiella aerogenes': 'Klebsiella aerogenes',
    'klebsiella oxytoca': 'Klebsiella oxytoca',
    'klebsiella pneumoniae group': 'Klebsiella pneumoniae',
    'proteus spp': 'Proteus species',
    'pseudomonas aeruginosa': 'Pseudomonas aeruginosa',
    'serratia marcescens': 'Serratia marcescens',
    'staphylococcus aureus': 'Staphylococcus aureus',
    'streptococcus agalactiae': 'Streptococcus agalactiae',
    'streptococcus pneumoniae': 'Streptococcus pneumoniae',
    'covid_19': 'SARS-CoV-2',
    'moraxella catarrhalis': 'Moraxella catarrhalis',
    'legionalla pneumophilia': 'Legionella pneumophilia',
    'streptococcus pyogenes': 'Streptococcus pyogenes',
    'Achromobacter denitrificans': 'Achromobacter species',
    'Achromobacter xylosoxidans': 'Achromobacter species',
    'Acinetobacter baumannii complex': 'Acinetobacter baumannii',
    'acinetobacter calcoaceticus-baumannii complex': 'Acinetobacter baumannii',
    'Corynebacterium striatum': 'Corynebacterium species',
    'Elizabethkingia meningoseptica': 'Elizabethkingia species',
    'Enterobacter cloacae complex': 'Enterobacter cloacae',
    'Granulicatella adiacens': 'Granulicatella species',
    'Enterobacter aerogenes': 'Klebsiella aerogenes',
    'Proteus mirabilis': 'Proteus species',
    'Streptococcus agalactiae (Group B)': 'Streptococcus agalactiae',
    'Streptococcus pyogenes (Group A)': 'Streptococcus pyogenes',
    'Citrobacter freundii group': 'Citrobacter freundii group',
}

In [18]:
pathogen_groups = pd.read_csv(
    '../data/02_pathogens - Sheet1.csv'
)
pathogen_groups = pathogen_groups.iloc[2:, :].copy()
pathogen_groups.columns = ['name'] + pathogen_groups.columns[1:].tolist()
pathogen_groups.set_index('name', inplace=True)
pathogen_groups = pathogen_groups.loc[:, 'Which CFU cutoff to use if not default 1000?':].copy()
pathogen_groups.columns = [
    'cfu_cutoff', 'small_groups', 'medium_groups', 'big_groups', 'large_groups',
    'kingdom'
]
pathogen_groups = pathogen_groups.loc[pathogen_groups.small_groups.ne('Discard')].copy()
pathogen_groups.big_groups.unique()

array(['NF', 'OF', 'Other', 'GNR', 'ST', 'EC', 'Influenza', 'MRSA',
       'SARS-CoV-2', 'SA', 'CONS'], dtype=object)

In [19]:
def define_pathogens(row):
    val = row.Pathogen_results
    # Some BALs don't have PCR or Culture test at all!
    if not isinstance(val, str):
        return {'test_performed': False}
    val = json.loads(val)
    # Get % neutrophils for rules 4 & 5
    pct_neutrophils = row.BAL_pct_neutrophils
    # If PCR was performed (for Adam's certainty)
    pcr_performed = val['pcr']['performed']
    # By default `clear_cut` is set to if PCR performed:
    #   I would think that for pathogens tests without PCR
    #   it's not that solid
    clear_cut = pcr_performed

    # Accumulate binary columns here
    pathogens = set()
    # Some test performed (PCR or Culture)
    pathogens.add('test_performed')
    for vir in val['pcr']['virus']:
        # Add normalized names for viruses
        pathogens.add(PATHOGEN_NAMES.get(vir, vir))
    for bac in val['pcr']['bacteria']:
        # Detect MRSA in PCR
        if (bac == 'staphylococcus aureus'
                and 'meca/c and mrej' in val['pcr']['resistance']
        ):
            val['pcr']['bacteria'].append('MRSA')
            pathogens.add('MRSA')
        else:
            # Add normalized names for bacteria
            pathogens.add(PATHOGEN_NAMES.get(bac, bac))
    for p in val['culture']['organisms']:
        name = p['name']
        # Extract MRSA from Culture
        if name == 'Staphylococcus aureus' and p['resistance'] == 'mrsa':
            name = 'MRSA'
        name = PATHOGEN_NAMES.get(name, name)
        # Discard pathogens we agreed to discard in the google sheet
        if name not in pathogen_groups.index:
            continue
        # Get custom CFU cutoff from google sheet
        cfu_cutoff = pathogen_groups.loc[name, 'cfu_cutoff']
        if not isinstance(cfu_cutoff, str) or len(cfu_cutoff.strip()) == 0:
            cfu_cutoff = 1000  # by default use 1000
        else:
            cfu_cutoff = int(cfu_cutoff.strip())

        # Add pathogen to current list if above cutoff
        if p['cfu'] >= cfu_cutoff:
            pathogens.add(name)
        # Or, if below, but % neutrophils is high, add it too
        #    NB: not detected pathogens would not be in the
        #    list we're iterating over
        elif pct_neutrophils >= 50:
            pathogens.add(name)

        # Set Adam's clear_cut flag to false if Culture is at low level
        #    or if culture is at high level, but PCR is available and
        #    negative
        if p['cfu'] < 1000 or (
                pcr_performed
                and name in val['pcr']['tests']
                and name not in val['pcr']['bacteria']
        ):
            # The output with print statement looks good:
            #   only bacteria with <1000 CFUs pop up
            # print(f'Setting clear-cut to False because of {name} at {p["cfu"]}')
            clear_cut = False

    # If no pathogens disabled the clear_cut flag, add it
    if clear_cut:
        pathogens.add('clear-cut')

    # Now process groups up the pathogen tree from the google sheets
    #     'small_groups', 'medium_groups', 'big_groups', 'large_groups', 'kingdom'
    groups = set()
    for name in pathogens:
        if name in ('clear-cut', 'test_performed'):
            continue
        # If group 5 level defined, take it
        if isinstance(pathogen_groups.small_groups[name], str):
            groups.add(f'group5_{pathogen_groups.small_groups[name]}')
        # Otherwise take level 4 as level 5
        else:
            groups.add(f'group5_{pathogen_groups.medium_groups[name]}')
        groups.add(f'group4_{pathogen_groups.medium_groups[name]}')
        groups.add(f'group3_{pathogen_groups.big_groups[name]}')
        groups.add(f'group2_{pathogen_groups.large_groups[name]}')
        groups.add(f'group1_{pathogen_groups.kingdom[name]}')
    pathogens |= groups

    return {PATHOGEN_NAMES.get(x, x): True for x in pathogens}

In [20]:
bal_pathogens = pd.DataFrame(
    sc_labels.loc[
        :,
        ['BAL_pct_neutrophils', 'Pathogen_results']
    ].apply(define_pathogens, axis='columns').tolist()
).fillna(False)

In [21]:
NOT_PATHOGEN_COLUMNS = ('clear-cut', 'test_performed')
def combine_pathogens(row):
    result = [row.test_performed, row['clear-cut']]
    for level in range(6):
        if level == 0:
            cols = row.index[
                ~row.index.isin(NOT_PATHOGEN_COLUMNS)
                & ~row.index.str.startswith('group')
            ]
        else:
            cols = row.index[row.index.str.startswith(f'group{6 - level}_')]
        level_vals = row[cols]
        pos_vals = list(sorted(level_vals.index[level_vals].tolist()))
        if pos_vals:
            val = '; '.join(pos_vals)
        else:
            val = 'pathogen-negative'
        result.append(val)
    return pd.Series(
        result,
        index=['test_performed', 'clear_cut'] + [f'level_{i}' for i in range(6, 0, -1)]
    )

bal_pathogens_summary = bal_pathogens.apply(combine_pathogens, axis='columns')

In [22]:
bal_pathogens_summary

,test_performed,clear_cut,level_6,level_5,level_4,level_3,level_2,level_1
0,True,True,influenza a; influenza a h1 (seasonal),group5_Influenza,group4_Influenza,group3_Influenza,group2_Influenza,group1_Virus
1,True,True,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative
2,True,True,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative
3,True,True,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative
4,True,False,Chryseobacterium indologenes,group5_15,group4_ENVIR,group3_GNR,group2_Gram-,group1_Bacteria
...,...,...,...,...,...,...,...,...
296,False,False,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative
297,False,False,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative
298,False,False,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative
299,False,False,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative,pathogen-negative


In [23]:
sc_labels['Pathogens'] = bal_pathogens_summary.level_6.values
sc_labels.loc[~bal_pathogens_summary.test_performed.values, 'Pathogens'] = np.nan

In [24]:
METADATA_FIELDS = {
    'bal_barcode': 'Sample ID',
    'Pathogens': 'Pathogens',
    'perturbation_groups_2': 'Pathogen groups',
    'SOFA_score': 'SOFA score',
    'pathogen_groups': 'pathogen_groups',
    'days_on_ventilator': 'Days on ventilator (log)',
    'episode_type': 'Pneumonia episode type',
    'Binary_outcome': 'Mortality',
    'Discharge_disposition': 'Discharge disposition',
    'Immunocompromised_flag': 'Immunocompromised',
    'days_of_icu_abx_until_today': 'ICU antibiotic days to date',
    'cumulative_icu_steroid_dose_until_today': 'ICU steroid dose to date',
    'is_culture_negative_pneumonia': 'Pathogen negative',
    'is_episode_cured': 'VAP cure in 7 days',
    'Sex': 'Sex',
    'cohort': 'Cohort'
}

In [25]:
RARE_PATHOGENS = [
    'Gram-*; SARS-CoV-2',
    'Gram-*; SARS-CoV-2; Gram+',
    'Other viruses',
    'Gram-*; Pseudomonas aeruginosa',
    'Pseudomonas aeruginosa; Gram+',
    'Other viruses; Pseudomonas aeruginosa',
    'Gram-*; Other viruses; SARS-CoV-2'
]

In [26]:
sc_labels = sc_labels[METADATA_FIELDS.keys()].set_index('bal_barcode')

In [27]:
sc_labels = sc_labels.rename(columns=METADATA_FIELDS)

In [28]:
sc_labels.loc[sc_labels['Pathogen negative'].fillna(False), 'Sample group'] = 'Pathogen-negative pneumonia'

In [29]:
sc_labels.loc[sc_labels['pathogen_groups'].isin(RARE_PATHOGENS), 'Sample group'] = 'Rare pathogens'

In [30]:
sc_labels['Pathogen groups'] = sc_labels['Pathogen groups'].replace({
    'Pseudomonas aeruginosa': 'Pseudomonas',
    'Pseudomonas aeruginosa; SARS-CoV-2': 'SARS-CoV-2; Pseudomonas',
    'Gram-*': 'Other Gram–',
    'Gram-*; Gram+': 'Other Gram–; Gram+'
})

In [31]:
sc_labels.loc[sc_labels['Cohort'].eq('LongCOVID'), 'Sample group'] = 'PASC'
sc_labels.loc[sc_labels['Pathogen groups'].eq('Healthy'), 'Sample group'] = 'Healthy'
sc_labels.loc[sc_labels['Pathogen groups'].eq('NPC'), 'Sample group'] = 'NPC'
sc_labels.loc[sc_labels['Pathogen groups'].isin(['Early SARS-CoV-2', 'Late SARS-CoV-2']), 'Sample group'] = 'Viral'
sc_labels.loc[sc_labels['Pathogen groups'].isin(
    ['Pseudomonas', 'Gram+', 'Other Gram–', 'Other Gram–; Gram+']
), 'Sample group'] = 'Bacterial'
sc_labels.loc[sc_labels['Pathogen groups'].isin(
    ['SARS-CoV-2; Pseudomonas', 'Early SARS-CoV-2; Gram+', 'Late SARS-CoV-2; Gram+']
), 'Sample group'] = 'Mixed'
sc_labels.loc[
    (
        sc_labels['Pathogens'].eq('pathogen-negative')
        & sc_labels['Sample group'].ne('Pathogen-negative pneumonia')
        & sc_labels['Pathogen groups'].eq('discard')
        & sc_labels['Pneumonia episode type'].isin(['CAP', 'HAP', 'VAP', 'VVAP'])
    ),
    'Sample group'
] = 'Pathogen cleared'
sc_labels['Sample group'] = sc_labels['Sample group'].fillna('Other')

In [32]:
sc_labels['Sample group'].value_counts(dropna=False)

Viral                          60
Bacterial                      58
Mixed                          33
Pathogen-negative pneumonia    32
Other                          28
NPC                            26
PASC                           25
Rare pathogens                 22
Healthy                         9
Pathogen cleared                8
Name: Sample group, dtype: int64

In [33]:
sc_labels['Pathogen groups'] = sc_labels['Pathogen groups'].replace({'discard': np.nan})

In [34]:
sc_labels['NPC category'] = ''
npc_cats = pd.Series(sc_labels.index, index=sc_labels.index).map(common_data.NPC_CATEGORIES).dropna()
sc_labels.loc[
    npc_cats.index,
    'NPC category'
] = npc_cats

In [35]:
sc_labels.drop(columns=['Cohort', 'Pathogen negative', 'pathogen_groups'], inplace=True)

In [ ]:
all_samples = []
all_cell_types = []
for p in sorted(BASE.iterdir()):
    ct_name = p.name.replace('_', ' ')
    ct_name = common_plots.CELL_TYPES_DISPLAY.get(ct_name, ct_name)
    if not p.is_dir():
        continue
    print(f'Processing {ct_name}')
    if not (p / 'transformed.tsv').exists():
        print('Skipping, not matrix')
        continue

    pseudobulk = pd.read_table(p / 'transformed.tsv', delim_whitespace=True).T
    metadata = pd.read_csv(p / 'meta.csv', index_col=0)
    all_samples += list(metadata['sample'])
    pseudobulk = pseudobulk.loc[metadata['sample'], :]

    if pseudobulk.shape[0] < 50:
        print(f'Skipping, less than 50 samples')
        continue

    gsva = decoupler.run_gsva(
        pseudobulk,
        net=hallmark,
        source='geneset',
        target='genesymbol',
        kcdf=True,
        seed=1066
    ).T
    gsva.index = gsva.index.str.replace('HALLMARK_', '')
    gsva.round(5).to_csv(OUTDIR / f'{ct_name}_gsva.tsv', sep='\t')
    all_cell_types.append(ct_name)

Processing AT1 and AT2
Skipping, less than 50 samples
Processing B cells
Processing CD4 T cells
Processing CD8 T cells
Processing Ciliated cells
Processing Classical monocytes-1 CCR2
Processing Classical monocytes-2 IL1B
Processing DC1
Skipping, less than 50 samples
Processing DC2
Processing Hematopoietic stem cells
Skipping, less than 50 samples
Processing Ionocytes
Skipping, not matrix
Processing MRC1+C1QA+ AM
Processing MRC1+C1QA– AM
Processing Mast cells
Skipping, less than 50 samples
Processing Migratory DC
Skipping, less than 50 samples
Processing NUPR1+ AM
Processing Non-classical monocytes
Skipping, less than 50 samples
Processing Interstitial macrophages
Processing Plasma cells
Processing Proliferating CD4 T cells
Processing Proliferating CD8 T cells
Processing Proliferating NUPR1+ AM
Processing Proliferating γδT cells
Skipping, less than 50 samples
Processing Proliferating plasma cells
Processing Secretory cells
Skipping, less than 50 samples
Processing Tregs
Processing γδT c

In [37]:
metadata = sc_labels.loc[sc_labels.index.isin(all_samples)].copy()
metadata['Days on ventilator (log)'] = np.log2(metadata['Days on ventilator (log)'])
metadata['Flow-cytometry clusters'] = common_data.get_sc_categorical_covariates()[
    'Flow_clusters'
][metadata.index].values
metadata['Flow-cytometry clusters'] = 'Cluster ' + metadata['Flow-cytometry clusters'].astype(str)
metadata['scRNA-seq clusters'] = common_data.get_sc_categorical_covariates()[
    'ScRNAseq_clusters'
][metadata.index].values
metadata['scRNA-seq clusters'] = 'Cluster ' + metadata['scRNA-seq clusters'].astype(str)

In [38]:
metadata.Mortality = metadata.Mortality.replace({1: 'Yes', 0: 'No'})
metadata.Immunocompromised = metadata.Immunocompromised.replace({1: 'Yes', 0: 'No'})
metadata['VAP cure in 7 days'] = metadata['VAP cure in 7 days'].replace({1: 'Yes', 0: 'No', -1: np.nan})
metadata['Flow-cytometry clusters'] = metadata['Flow-cytometry clusters'].replace({'Cluster NA': np.nan})
metadata['scRNA-seq clusters'] = metadata['scRNA-seq clusters'].replace({'Cluster NA': np.nan})

In [39]:
CAT_NAMES = {
    'NI': 'Not infected',
    'Infection': 'Extrapulmonary infection',
    'Inflammation': 'Non-infectious alveolitis'
}
metadata['NPC category'] = metadata['NPC category'].replace(CAT_NAMES)

In [42]:
metadata.to_csv(OUTDIR / 'meta.tsv', sep='\t')

In [49]:
hallmark_pathways = pd.DataFrame(dict(id=hallmark.geneset.unique()))

In [50]:
hallmark_pathways.id = hallmark_pathways.id.str.replace('HALLMARK_', '')
hallmark_pathways['Gene set'] = hallmark_pathways.id.apply(pb_utils.process_hallmark_name)

In [51]:
hallmark_pathways.to_csv(OUTDIR / 'pathways.tsv', sep='\t')

In [53]:
all_cell_types

['B cells',
 'CD4 T cells',
 'CD8 T cells',
 'Ciliated cells',
 'Classical monocytes-1 CCR2',
 'Classical monocytes-2 IL1B',
 'DC2',
 'MRC1+C1QA+ AM',
 'MRC1+C1QA– AM',
 'NUPR1+ AM',
 'Interstitial macrophages',
 'Plasma cells',
 'Proliferating CD4 T cells',
 'Proliferating CD8 T cells',
 'Proliferating NUPR1+ AM',
 'Proliferating plasma cells',
 'Tregs',
 'γδT cells']